In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np
import re
import random

In [2]:
current_path = Path().resolve() 
scenarios_path = current_path.parent.parent / "scenarios_inputs" / "nie" 
annotated_output_path = current_path.parent.parent / "annotated_outputs" / "nie"

print(f"scenarios_path: {scenarios_path}")
print(f"annotated_output_path: {annotated_output_path}")

utils_path = current_path.parent.parent / "src"
print(f"utils_path: {utils_path}")
sys.path.insert(0, str(utils_path))
import utils
print(utils.__file__)

scenarios_path: C:\Users\rylen\Dropbox\2025_moral_scenario_annotation\code\rylen\scenarios_inputs\nie
annotated_output_path: C:\Users\rylen\Dropbox\2025_moral_scenario_annotation\code\rylen\annotated_outputs\nie
utils_path: C:\Users\rylen\Dropbox\2025_moral_scenario_annotation\code\rylen\src
C:\Users\rylen\Dropbox\2025_moral_scenario_annotation\code\rylen\src\utils.py


Utility Functions (LLM Use)

In [5]:
def pluralize_entities(entity: str) -> dict:
    system_prompt_content = """""You are an expert in English grammar. Modify the given entity so that it is written in its plural form without any numbers included. Return a json called 'converted_text' with the converted text only."""
    user_prompt_content = entity
    GPT_response = utils.get_response_dict(system_prompt_content, user_prompt_content)
    return GPT_response["converted_text"]

def convert_to_Ziv(scenario_text: str) -> dict:
    system_prompt_content = """""You are an expert in English grammar. Rewrite the following text so that it is written from the perspective of a character name Ziv in third person instead of being written in the first person. Replace every instance of the first person pronoun (I, me, my, etc) with either the name Ziv or the pronouns they, their, them, etc. Return a json called 'converted text' with the converted text only."""
    user_prompt_content = scenario_text
    GPT_response = utils.get_response_dict(system_prompt_content, user_prompt_content)
    return GPT_response["converted_text"]

In [7]:
# --- Load outcomes ---
temp_df = pd.read_csv(current_path / "nie_primary_outcomes.csv", encoding="utf-8")
primary_outcome_idx = dict(zip(temp_df["id"], temp_df["outcome"]))

# --- Load master scenario list ---
with open(scenarios_path / "nie_scenarios.json", "r", encoding="utf-8") as f:
    master_data = json.load(f)

master_by_id = {
    entry["id"]: entry
    for entry in master_data
    if entry["id"] in primary_outcome_idx
}

# --- Regex for entity files ---
pattern = re.compile(r"nie_scenarios_(\d+)_choice_1\.json")

results = []
max_entities = 0

# --- Pass 1: Extract everything and track max entity count ---
for file in annotated_output_path.glob("*.json"):
    match = pattern.match(file.name)
    if not match:
        continue

    scenario_id = int(match.group(1))

    if scenario_id not in master_by_id:
        continue

    scenario_info = master_by_id[scenario_id]
    scenario_text = scenario_info["text"]
    action = scenario_info["options"].get("1", None)
    chosen_outcome = primary_outcome_idx.get(scenario_id, None)

    # Load entity file
    entities = []
    with open(file, "r", encoding="utf-8") as f:
        previously_changed = []
        for line in f:
            line = line.strip()
            if not line:
                continue

            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue  # Skip malformed lines

            node = obj.get("node", {})
            if node.get("kind") == "being":
                label = node.get("label", "").strip()

                # Exclude "I"
                if label and label.lower() == "i":
                    continue
        
                m = re.fullmatch(r"(.+?)\s+(\d+)", label.lower())
                if m:
                    base = m.group(1).strip()
                    if base[:-2] not in previously_changed:
                        normalized = pluralize_entities(base)
                        previously_changed.append(normalized)
                else:
                    normalized = label
                
                if normalized not in entities:
                    entities.append(normalized)


    max_entities = max(max_entities, len(entities))

    results.append({
        "id": scenario_id,
        "scenario_text": scenario_text,
        "action_choice": action,
        "outcome": chosen_outcome,
        "entities": entities
    })

# --- Pass 2: Expand entity columns ---
expanded_rows = []
for r in results:
    row = {
        "id": r["id"],
        "scenario_text": r["scenario_text"],
        "action_choice": r["action_choice"],
        "outcome": r["outcome"],
    }

    # Fill entity_i columns
    for i in range(max_entities):
        row[f"entity_{i+1}"] = r["entities"][i] if i < len(r["entities"]) else ""

    expanded_rows.append(row)

df = pd.DataFrame(expanded_rows)

# --- Pairing logic ---
pairs = {}
for r in expanded_rows:
    pair_id = r["id"] // 2
    pairs.setdefault(pair_id, []).append(r)

bucket_A, bucket_B = [], []

for items in pairs.values():
    random.shuffle(items)
    if len(items) >= 1:
        bucket_A.append(items[0])
    if len(items) >= 2:
        bucket_B.append(items[1])

# --- Save ---
pd.DataFrame(bucket_A).to_csv("nie_survey_set_A.csv", index=False)
pd.DataFrame(bucket_B).to_csv("nie_survey_set_B.csv", index=False)

railroad workers
railroad workers
railroad workers
people
people
people
people
people
people
people
people
people
people
neighbors
neighbors
neighbors
neighbors
neighbors
neighbors
neighbors
neighbors
neighbors
neighbors
enemy soldiers
enemy soldiers
neighbors
neighbors
neighbors
neighbors
neighbors
neighbors
neighbors
neighbors
neighbors
neighbors
enemy soldiers
enemy soldiers
crewmembers
crewmembers
crewmembers
crewmembers
crewmembers
crewmembers
crewmembers
crewmembers
crewmembers
crewmembers
divers
divers
divers
divers
divers
divers
divers
divers
divers
sharks
sharks
sharks
sharks
sharks
divers
divers
divers
divers
divers
divers
divers
divers
divers
sharks
sharks
sharks
sharks
sharks
orphans
orphans
orphans
orphans
orphans
orphans
orphans
orphans
orphans
orphans
orphans
orphans
orphans
orphans
orphans
orphans
orphans
orphans
orphans
orphans
orphans
orphans
children
children
children
patients
patients
patients
patients
patients
patients
patients
patients
patients
patients
crewmember